# BiCGSTAB — Interactive Numerical Experiments

Compare right-preconditioned BiCGSTAB with SciPy and explore
nonmonotonic convergence on nonsymmetric systems.

**Instructions:** Select the parameters and click an experiment
button to display its numerical results and convergence plots.


In [ ]:
from pathlib import Path
import sys
import time

# Discover the checkout root in local Jupyter and Binder sessions.
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "krylov_methods").is_dir():
        root = candidate
        break
else:
    raise RuntimeError("Launch Jupyter from inside the repository checkout.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from krylov_methods.notebooks.notebook_utils import (
    spd_problem, nonsymmetric_problem, residual_plot, error_plot, report,
)

from scipy.sparse.linalg import bicgstab as scipy_bicgstab
from krylov_methods.bicgstab_solver import solve_bicgstab

from scipy.sparse.linalg import LinearOperator
from krylov_methods.preconditioners import make_preconditioner


In [ ]:
system = widgets.Dropdown(
    options=[("Nonsymmetric", "nonsymmetric"), ("Real SPD", "spd")],
    description="System",
)
size = widgets.IntSlider(value=30, min=5, max=100, step=5, description="Size n")
condition = widgets.FloatLogSlider(value=1e4, base=10, min=1, max=6, step=1, description="Cond(A), SPD")
tol = widgets.FloatLogSlider(value=1e-9, base=10, min=-12, max=-5, step=1, description="rtol")
budget = widgets.IntSlider(value=200, min=20, max=600, step=20, description="Max iterations")
preconds = widgets.Dropdown(
    options=["none", "jacobi", "ilu", "lu", "ssor", "cholesky", "ic0", "custom (diagonal)"],
    value="none",
    description="Precond",
    layout=widgets.Layout(width="350px"),
)
button = widgets.Button(description="Run experiment", button_style="success")
output = widgets.Output()
bicg_residual_output = widgets.Output()
bicg_error_output = widgets.Output()


def run(_):
    with output:
        clear_output(wait=True)
        bicg_residual_output.clear_output(wait=True)
        bicg_error_output.clear_output(wait=True)
        A, b, x_true = (
            nonsymmetric_problem(size.value)
            if system.value == "nonsymmetric"
            else spd_problem(size.value, condition.value)
        )
        threshold = max(1e-12, tol.value * np.linalg.norm(b))
        eigs = np.linalg.eigvals(A)
        print("SYSTEM INFORMATION")
        print(f"Matrix: {system.value}; dimension={size.value}; dense; dtype={A.dtype}")
        print(f"Symmetry/Hermitian error ||A-Aᴴ||₂={np.linalg.norm(A-A.conj().T, ord=2):.3e}")
        print(f"Condition number κ₂(A)={np.linalg.cond(A):.3e}")
        print(f"Eigenvalue real parts: [{eigs.real.min():.3e}, {eigs.real.max():.3e}]")
        print(f"Eigenvalue imaginary parts: [{eigs.imag.min():.3e}, {eigs.imag.max():.3e}]")
        print(f"||b||₂={np.linalg.norm(b):.3e}; ||x*||₂={np.linalg.norm(x_true):.3e}")
        print(f"Initial guess x₀=0; initial residual={np.linalg.norm(b):.3e}")
        print(f"True-residual stopping threshold={threshold:.3e}")
        print(f"Iteration budget={budget.value}; BiCGSTAB uses RIGHT preconditioning.\n")

        histories = {}
        error_histories = {}
        for name in (preconds.value,):
            if name in ("cholesky", "ic0") and system.value != "spd":
                print(f"Skipping {name}: requires a real SPD system.")
                continue
            custom = name == "custom (diagonal)"
            p = "custom" if custom else None if name == "none" else name
            M = (lambda v: v / np.diag(A)) if custom else None

            start = time.perf_counter()
            result = solve_bicgstab(
                A, b, rtol=tol.value, atol=1e-12, max_iter=budget.value,
                preconditioner=p, M=M, store_history=True,
            )
            elapsed = time.perf_counter() - start
            report("Our BiCG: " + name, result.solution, x_true, b, A,
                   result.iterations, result.converged)
            print(f"  Solver-call time={elapsed:.6f} s; status={result.message}")
            if result.breakdown:
                print("  Breakdown:", result.message)
            histories["BiCGSTAB / " + name] = result.residual_norms
            error_histories["BiCGSTAB / " + name] = [
                np.linalg.norm(xk - x_true) / np.linalg.norm(x_true)
                for xk in result.history
            ]

        # Compare SciPy with each selected preconditioner, not only None.
        for name in (preconds.value,):
            if name in ("cholesky", "ic0") and system.value != "spd":
                continue
            custom = name == "custom (diagonal)"
            kind = "custom" if custom else None if name == "none" else name
            custom_M = (lambda v: v / np.diag(A)) if custom else None
            inverse_action = make_preconditioner(A, kind, M=custom_M)
            scipy_M = LinearOperator(A.shape, matvec=inverse_action, dtype=A.dtype)
            scipy_history = [np.linalg.norm(b)]
            scipy_errors = [1.0]

            def scipy_callback(xk):
                scipy_history.append(np.linalg.norm(b - A @ xk))
                scipy_errors.append(np.linalg.norm(xk - x_true) / np.linalg.norm(x_true))

            start = time.perf_counter()
            x, info = scipy_bicgstab(
                A, b, M=scipy_M, rtol=tol.value, atol=1e-12,
                maxiter=budget.value, callback=scipy_callback,
            )
            elapsed = time.perf_counter() - start
            final_norm = np.linalg.norm(b - A @ x)
            # SciPy may converge with an alpha-only final step; its callback
            # is not guaranteed to run for that last update.
            if not np.isclose(scipy_history[-1], final_norm,
                              rtol=1e-12, atol=1e-14):
                scipy_history.append(final_norm)
                scipy_errors.append(np.linalg.norm(x-x_true) / np.linalg.norm(x_true))
            report("SciPy: " + name, x, x_true, b, A,
                   len(scipy_history)-1, info == 0)
            print(f"  SciPy time={elapsed:.6f} s; info={info}")
            histories["SciPy / " + name] = scipy_history
            error_histories["SciPy / " + name] = scipy_errors

        with bicg_residual_output:
            residual_plot(histories, threshold, "BiCGSTAB — True residual convergence", figsize=(4.1, 3.0))
        with bicg_error_output:
            error_plot(error_histories, "BiCGSTAB — Relative solution-error convergence", figsize=(4.1, 3.0))
        print("Interpretation: residual curves may oscillate. A small residual "
              "does not always imply small forward error when A is ill-conditioned.")
        print("Timing includes built-in preconditioner setup and iteration costs.")


button.on_click(run)


In [ ]:
nonnormality = widgets.FloatSlider(
    value=3.0, min=0.5, max=8.0, step=0.5, description="Upper coupling"
)
nonmonotonic_button = widgets.Button(description="Explore nonmonotonic convergence", button_style="info")
nonmonotonic_output = widgets.Output()
nonmonotonic_residual_output = widgets.Output()
nonmonotonic_error_output = widgets.Output()

def run_nonmonotonic(_):
    with nonmonotonic_output:
        clear_output(wait=True)
        nonmonotonic_residual_output.clear_output(wait=True)
        nonmonotonic_error_output.clear_output(wait=True)
        n = 25
        A = 2 * np.eye(n) + nonnormality.value * np.eye(n, k=1)
        A += 0.3 * np.eye(n, k=-1)
        x_true = np.ones(n)
        b = A @ x_true
        result = solve_bicgstab(A, b, max_iter=150, rtol=1e-10,
                                atol=1e-12, store_history=True)
        residuals = np.asarray(result.residual_norms)
        rises = int(np.count_nonzero(np.diff(residuals) > 0))
        print(f"Upper coupling={nonnormality.value:g}; cond₂(A)={np.linalg.cond(A):.3e}")
        report("BiCGSTAB", result.solution, x_true, b, A,
               result.iterations, result.converged)
        print(f"Residual increases observed: {rises}; status: {result.message}")
        if result.breakdown:
            print("Numerical breakdown detected.")
        with nonmonotonic_residual_output:
            residual_plot({"BiCGSTAB": residuals},
                          max(1e-12, 1e-10*np.linalg.norm(b)),
                          "BiCGSTAB residual fluctuations on a nonnormal system", figsize=(4.1, 3.0))
        errors = [np.linalg.norm(xk-x_true)/np.linalg.norm(x_true)
                  for xk in result.history]
        with nonmonotonic_error_output:
            error_plot({"BiCGSTAB": errors}, "BiCGSTAB forward error on a nonnormal system", figsize=(4.1, 3.0))

nonmonotonic_button.on_click(run_nonmonotonic)


In [ ]:

# -------------------------------------------------------
# BiCGSTAB — Compact interactive dashboard
# -------------------------------------------------------

bicg_status = widgets.HTML()
nonmonotonic_status = widgets.HTML()

bicg_parameters = (
    system, size, condition, tol, budget, preconds
)
nonmonotonic_parameters = (nonnormality,)

last_runs = {
    "bicgstab": None,
    "nonmonotonic": None,
}


def parameter_values(parameters):
    return tuple(widget.value for widget in parameters)


def update_status(name, parameters, indicator):
    previous = last_runs[name]

    if previous is None:
        indicator.value = ""
    elif parameter_values(parameters) == previous:
        indicator.value = (
            "<span style='color:#22863a;font-weight:600;'>"
            "● Simulation up to date</span>"
        )
    else:
        indicator.value = (
            "<span style='color:#b36b00;font-weight:600;'>"
            "● Parameters changed — rerun required</span>"
        )


def refresh_bicg(change=None):
    update_status("bicgstab", bicg_parameters, bicg_status)


def refresh_nonmonotonic(change=None):
    update_status(
        "nonmonotonic",
        nonmonotonic_parameters,
        nonmonotonic_status,
    )


for parameter in bicg_parameters:
    parameter.observe(refresh_bicg, names="value")

nonnormality.observe(refresh_nonmonotonic, names="value")


# -------------------------------------------------------
# Plot panels
# -------------------------------------------------------

def plot_panel(title, plot_output):
    return widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-weight:600;text-align:center;'>"
                + title + "</div>"
            ),
            plot_output,
        ],
        layout=widgets.Layout(
            flex="1 1 240px",
            min_width="220px",
            max_width="100%",
            padding="4px",
            overflow="hidden",
        ),
    )


bicg_residual_panel = plot_panel(
    "BiCGSTAB True Residual",
    bicg_residual_output,
)

bicg_error_panel = plot_panel(
    "BiCGSTAB Solution Error",
    bicg_error_output,
)

nonmonotonic_residual_panel = plot_panel(
    "Nonmonotonic Residual",
    nonmonotonic_residual_output,
)

nonmonotonic_error_panel = plot_panel(
    "Nonmonotonic Solution Error",
    nonmonotonic_error_output,
)

plot_panels = (
    bicg_residual_panel,
    bicg_error_panel,
    nonmonotonic_residual_panel,
    nonmonotonic_error_panel,
)

for panel in plot_panels:
    panel.layout.display = "none"

plots_row = widgets.HBox(
    list(plot_panels),
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="5px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Expandable numerical reports
# -------------------------------------------------------

bicg_details = widgets.Accordion(children=[output])
bicg_details.set_title(
    0, "BiCGSTAB system information & solver details"
)
bicg_details.selected_index = None

nonmonotonic_details = widgets.Accordion(
    children=[nonmonotonic_output]
)
nonmonotonic_details.set_title(
    0, "Nonmonotonic convergence details"
)
nonmonotonic_details.selected_index = None

bicg_details.layout.display = "none"
nonmonotonic_details.layout.display = "none"

bicg_details.layout.flex = "1 1 500px"
nonmonotonic_details.layout.flex = "1 1 500px"


# -------------------------------------------------------
# Reveal results after execution
# -------------------------------------------------------

def show_bicg_results(change=None):
    # The existing numerical callback executes first.
    bicg_details.layout.display = ""

    valid = not (
        system.value != "spd"
        and preconds.value in ("cholesky", "ic0")
    )

    if valid:
        bicg_residual_panel.layout.display = ""
        bicg_error_panel.layout.display = ""
        last_runs["bicgstab"] = parameter_values(
            bicg_parameters
        )
    else:
        bicg_residual_panel.layout.display = "none"
        bicg_error_panel.layout.display = "none"
        last_runs["bicgstab"] = None

    refresh_bicg()


def show_nonmonotonic_results(change=None):
    nonmonotonic_residual_panel.layout.display = ""
    nonmonotonic_error_panel.layout.display = ""
    nonmonotonic_details.layout.display = ""

    last_runs["nonmonotonic"] = parameter_values(
        nonmonotonic_parameters
    )
    refresh_nonmonotonic()


# Register presentation callbacks after numerical callbacks.
button.on_click(show_bicg_results)
nonmonotonic_button.on_click(show_nonmonotonic_results)


# -------------------------------------------------------
# Experiment controls
# -------------------------------------------------------

bicg_controls = widgets.VBox(
    [
        widgets.HTML("<h3>BiCGSTAB Convergence</h3>"),
        system,
        size,
        condition,
        tol,
        budget,
        preconds,
        button,
        bicg_status,
    ],
    layout=widgets.Layout(
        flex="1 1 450px",
        min_width="300px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

nonmonotonic_controls = widgets.VBox(
    [
        widgets.HTML("<h3>Nonmonotonic Convergence</h3>"),
        nonnormality,
        nonmonotonic_button,
        nonmonotonic_status,
    ],
    layout=widgets.Layout(
        flex="1 1 450px",
        min_width="280px",
        padding="10px",
        border="1px solid #dddddd",
    ),
)

controls_row = widgets.HBox(
    [bicg_controls, nonmonotonic_controls],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="stretch",
    ),
)

details_row = widgets.HBox(
    [bicg_details, nonmonotonic_details],
    layout=widgets.Layout(
        width="100%",
        flex_flow="row wrap",
        gap="10px",
        align_items="flex-start",
    ),
)


# -------------------------------------------------------
# Final dashboard
# -------------------------------------------------------

bicgstab_dashboard = widgets.VBox(
    [
        widgets.HTML("""
        <style>
        .bicgstab-compact-dashboard .output_area img,
        .bicgstab-compact-dashboard .jp-OutputArea-output img {
            max-width: 100% !important;
            height: auto !important;
        }
        </style>
        """),
        controls_row,
        plots_row,
        details_row,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="12px",
    ),
)

bicgstab_dashboard.add_class(
    "bicgstab-compact-dashboard"
)

display(bicgstab_dashboard)

# No automatic simulations.
# Results appear only after clicking experiment buttons.
